# 🖼️ From Noise to Masterpiece: CFG, DDIM & System Design

> **Chapter 09 — Text-to-Image Generation**  
> *Sampling strategies, evaluation metrics, and production system design* 🚀

## 🎛️ Classifier-Free Guidance (CFG)

### The Problem: Diversity vs. Quality

A diffusion model trained purely on text conditioning is like a student who hedges all their answers. When you ask for "a red sports car", it might produce a pinkish car-ish blob — plausible, but boring. It's averaging over all possible red sports cars rather than committing.

### The Solution: Amplify the Conditioning Signal 📢

**Classifier-Free Guidance (Ho & Salimans, 2022)** solves this with a clever trick:

**Training**: randomly drop the text conditioning with probability p (e.g., p=0.1). This forces the model to learn *two things simultaneously*:
- $\varepsilon_\theta(\mathbf{x}_t, c)$ — conditioned on text c
- $\varepsilon_\theta(\mathbf{x}_t, \emptyset)$ — unconditional (null embedding)

**Inference**: push the generation *away* from the unconditional direction and *toward* the conditional:

$$\boxed{\hat{\varepsilon} = \varepsilon_\theta(\mathbf{x}_t, \emptyset) + w \cdot \Big(\varepsilon_\theta(\mathbf{x}_t, c) - \varepsilon_\theta(\mathbf{x}_t, \emptyset)\Big)}$$

The **guidance scale w** is the key knob:

| w value | Behavior |
|---------|----------|
| w = 0 | Purely unconditional — ignores your prompt entirely |
| w = 1 | Standard conditional sampling — follows prompt loosely |
| w = 7.5 | Typical SD default — crisp, prompt-faithful images |
| w = 15+ | Over-saturated, bizarre, "too committed" to the prompt |

> 💡 **Intuition**: Think of the unconditional prediction as "what's average" and the conditional as "what you specifically asked for." CFG says: give me more of the specific, less of the average. The strength of "more" is w.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

# ── Toy 2D latent space to illustrate CFG geometry ───────────────────────────
# Imagine the 2D plane is the latent space of noise predictions.
# Unconditional prediction = "average blob"
# Conditional prediction = "what the text says"

eps_uncond = np.array([0.2, 0.1])      # ε_uncond  (baseline, near center)
eps_cond   = np.array([0.8, 0.7])      # ε_cond    (text-driven direction)

guidance_weights = [0, 1, 3, 7.5, 15]
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

def cfg_formula(eps_uncond, eps_cond, w):
    return eps_uncond + w * (eps_cond - eps_uncond)

guided_preds = [cfg_formula(eps_uncond, eps_cond, w) for w in guidance_weights]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d1117')

# ── Left: 2D vector visualization ────────────────────────────────────────────
ax1.set_xlim(-0.5, 3.5)
ax1.set_ylim(-0.5, 3.5)
ax1.set_title('CFG: Geometric View in Latent Space', color='white', fontsize=11)
ax1.tick_params(colors='white')
ax1.set_xlabel('Noise dim 1', color='white')
ax1.set_ylabel('Noise dim 2', color='white')

# Draw axes
ax1.axhline(0, color='#333', linewidth=0.5)
ax1.axvline(0, color='#333', linewidth=0.5)

# Plot unconditional and conditional endpoints
ax1.scatter(*eps_uncond, color='#aaa', s=100, zorder=5)
ax1.annotate('ε_uncond\n("average")', eps_uncond, textcoords='offset points',
             xytext=(-30, -35), color='#aaa', fontsize=9)
ax1.scatter(*eps_cond, color='#f0c040', s=100, zorder=5)
ax1.annotate('ε_cond\n("text says")', eps_cond, textcoords='offset points',
             xytext=(5, 5), color='#f0c040', fontsize=9)

# Draw the guidance vectors for each w
for w, gp, color in zip(guidance_weights, guided_preds, colors):
    ax1.annotate('', xy=gp, xytext=eps_uncond,
                 arrowprops=dict(arrowstyle='->', color=color, lw=2.0))
    ax1.scatter(*gp, color=color, s=80, zorder=6)
    ax1.annotate(f'w={w}', gp, textcoords='offset points',
                 xytext=(5, 3), color=color, fontsize=8)

legend_patches = [mpatches.Patch(color=c, label=f'w={w}') for c, w in zip(colors, guidance_weights)]
ax1.legend(handles=legend_patches, loc='upper left',
           facecolor='#1a1a2e', labelcolor='white', fontsize=8)

# ── Right: w vs output magnitude / quality-diversity tradeoff ─────────────────
w_range = np.linspace(0, 20, 200)
guided_norms = [np.linalg.norm(cfg_formula(eps_uncond, eps_cond, w)) for w in w_range]
# Simulate FID-like score (quality peaks around w=7, diversity drops)
quality    = np.exp(-((w_range - 7.5) ** 2) / 20)  # peaks at w=7.5
diversity  = np.exp(-w_range / 5)                    # decreases with w

ax2.set_title('Guidance Scale w: Quality vs Diversity Tradeoff', color='white', fontsize=11)
ax2.tick_params(colors='white')
ax2.set_xlabel('Guidance scale w', color='white')
ax2.set_ylabel('Normalized score', color='white')

ax2.plot(w_range, quality,   color='#2ecc71', lw=2.5, label='Quality (FID-like)')
ax2.plot(w_range, diversity, color='#e74c3c', lw=2.5, label='Diversity')
ax2.axvline(7.5, color='#f0c040', lw=1.5, linestyle='--', label='SD default (w=7.5)')
ax2.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)

# Shade sweet spot
ax2.axvspan(5, 10, alpha=0.1, color='#f0c040')
ax2.text(7.5, 0.05, 'sweet\nspot', ha='center', color='#f0c040', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/cfg_visualization.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print("📐 CFG Formula: ε̂ = ε_uncond + w · (ε_cond - ε_uncond)")
print()
print(f"   {'w':>5} | {'ε̂ vector':>20} | {'||ε̂||':>8}")
print("   " + "-" * 42)
for w, gp in zip(guidance_weights, guided_preds):
    print(f"   {w:>5} | ({gp[0]:>7.3f}, {gp[1]:>7.3f}) | {np.linalg.norm(gp):>8.3f}")
print()
print("✅ w=0 → pure unconditional; w=1 → exactly ε_cond; w>1 → extrapolate!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(0)

# ── Noise schedule (standard DDPM) ───────────────────────────────────────────
T_DDPM = 1000
betas   = np.linspace(1e-4, 0.02, T_DDPM)
alphas  = 1.0 - betas
alpha_bar_np = np.cumprod(alphas)

# ── DDPM sampling step: x_{t-1} from x_t (one step) ─────────────────────────
def ddpm_step(
    x_t: np.ndarray,
    t: int,
    eps_pred: np.ndarray,
    add_noise: bool = True,
) -> np.ndarray:
    """
    DDPM reverse step (Ho et al. 2020, Algorithm 2):
      x_{t-1} = 1/sqrt(α_t) * (x_t - β_t/sqrt(1-ᾱ_t) * ε̂) + σ_t * z
    """
    a_t  = alphas[t - 1]
    ab_t = alpha_bar_np[t - 1]
    b_t  = betas[t - 1]

    # Compute mean
    coeff = b_t / np.sqrt(1.0 - ab_t)
    mean  = (1.0 / np.sqrt(a_t)) * (x_t - coeff * eps_pred)

    if add_noise and t > 1:
        sigma = np.sqrt(b_t)
        mean += sigma * np.random.randn(*x_t.shape)
    return mean


# ── DDIM sampling step (Song et al. 2020, Eq. 12) ────────────────────────────
def ddim_step(
    x_t: np.ndarray,
    t: int,
    t_prev: int,
    eps_pred: np.ndarray,
    eta: float = 0.0,       # η=0 → deterministic DDIM
) -> np.ndarray:
    """
    DDIM reverse step (deterministic when η=0):
      x̂₀  = (x_t - sqrt(1-ᾱ_t) * ε̂) / sqrt(ᾱ_t)          # predicted x0
      x_{t-1} = sqrt(ᾱ_{t-1}) * x̂₀ + sqrt(1-ᾱ_{t-1}) * ε̂  # DDIM update
    """
    ab_t    = alpha_bar_np[t - 1]
    ab_prev = alpha_bar_np[t_prev - 1] if t_prev > 0 else 1.0

    # Predicted clean image
    x0_pred = (x_t - np.sqrt(1.0 - ab_t) * eps_pred) / np.sqrt(ab_t)
    x0_pred = np.clip(x0_pred, -1.5, 1.5)   # stabilize

    # DDIM direction toward x_t
    dir_xt = np.sqrt(1.0 - ab_prev) * eps_pred
    x_prev = np.sqrt(ab_prev) * x0_pred + dir_xt
    return x_prev


# ── Simulate both samplers on a 1D signal (easier to visualize) ───────────────
SHAPE = (32, 32)

def mock_eps_predictor(x_t, t):
    """Mock noise predictor: returns a slightly de-noised signal."""
    ab = alpha_bar_np[t - 1]
    # Perfect predictor would return the true ε; ours returns a noisy approximation
    true_eps = (x_t - np.sqrt(ab) * 0.5) / np.sqrt(1.0 - ab + 1e-8)
    return true_eps + 0.05 * np.random.randn(*x_t.shape)


def run_ddpm(steps: int = 1000) -> tuple[list, float]:
    np.random.seed(1)
    x = np.random.randn(*SHAPE)
    trajectory = [x.copy()]
    t_start = time.time()
    step_indices = list(range(steps, 0, -1))
    for t in step_indices:
        eps_hat = mock_eps_predictor(x, t)
        x = ddpm_step(x, t, eps_hat, add_noise=True)
        if t % (steps // 10) == 0 or t == 1:
            trajectory.append(x.copy())
    return trajectory, time.time() - t_start


def run_ddim(steps: int = 20) -> tuple[list, float]:
    np.random.seed(1)
    x = np.random.randn(*SHAPE)
    # Uniformly spaced timesteps from T to 0
    ts = np.linspace(T_DDPM, 1, steps + 1).astype(int)
    trajectory = [x.copy()]
    t_start = time.time()
    for i in range(len(ts) - 1):
        t, t_prev = int(ts[i]), int(ts[i + 1])
        eps_hat = mock_eps_predictor(x, t)
        x = ddim_step(x, t, t_prev, eps_hat, eta=0.0)
        trajectory.append(x.copy())
    return trajectory, time.time() - t_start


print("⏱️  Running DDPM (1000 steps) and DDIM (20 steps)...")
ddpm_traj, ddpm_time = run_ddpm(steps=1000)
ddim_traj, ddim_time = run_ddim(steps=20)
print(f"   DDPM: {len(ddpm_traj)-1} NFE (network forward evals) — {ddpm_time:.3f}s")
print(f"   DDIM:  {len(ddim_traj)-1} NFE                         — {ddim_time:.3f}s")
print()

# ── Visualize trajectories ────────────────────────────────────────────────────
n_show = 11
fig, axes = plt.subplots(2, n_show, figsize=(2.0 * n_show, 5))
fig.patch.set_facecolor('#1a1a2e')

ddpm_show = ddpm_traj[:n_show]
ddim_show = ddim_traj[:n_show]

for col in range(n_show):
    for row, (traj, label) in enumerate([(ddpm_show, 'DDPM'), (ddim_show, 'DDIM')]):
        if col < len(traj):
            axes[row, col].imshow(traj[col], cmap='plasma', vmin=-2.5, vmax=2.5)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(label, color='white', fontsize=10)

fig.suptitle(
    '⚡ DDPM (1000 steps, stochastic) vs DDIM (20 steps, deterministic)',
    color='white', fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/tmp/ddpm_vs_ddim.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print("📊 Key Differences:")
print("   DDPM | Stochastic (adds random noise each step) | 1000 NFE | Higher diversity")
print("   DDIM | Deterministic (η=0)                       |   20 NFE | Same seed = same image")
print()
print("   DDIM speedup: ~50× fewer model evaluations!")
print("   Trade-off: slightly lower diversity, but often similar or better quality.")

## 📏 CLIPScore — Measuring Text-Image Alignment

### The Evaluation Challenge

How do you score whether a generated image actually matches its text prompt? Human evaluation is expensive and slow. FID (Fréchet Inception Distance) measures image quality vs. a reference dataset but doesn't measure *prompt faithfulness*.

**CLIPScore** (Hessel et al. 2021) uses OpenAI's CLIP model to measure alignment:

$$\text{CLIPScore}(c, I) = w \cdot \max\!\left(0,\ \cos\!\left(\mathbf{e}_c,\ \mathbf{e}_I\right)\right)$$

Where:
- $\mathbf{e}_c$ = CLIP text embedding of the prompt c
- $\mathbf{e}_I$ = CLIP image embedding of the generated image I
- $\cos(\cdot, \cdot)$ = cosine similarity
- $w = 2.5$ is a scaling constant

### Why CLIP? 🔍

CLIP was trained on 400M (image, text) pairs with contrastive learning — it learned to align image and text representations in a shared embedding space. If a generated image of a "golden retriever" has an embedding close to the text embedding of "golden retriever", CLIP gives it a high score.

### Limitations ⚠️

| Limitation | Example |
|-----------|--------|
| Bag-of-words bias | "A cat chasing a dog" ≈ "A dog chasing a cat" (CLIP can't tell) |
| Counting failure | "Three apples" vs "Two apples" scores similarly |
| Spatial relations | "Object on the left" is hard for CLIP to verify |
| Style insensitive | "Impressionist painting" vs "photo" may score the same |

> 💡 Use CLIPScore alongside FID and human eval — no single metric tells the whole story.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ── Simulate CLIP embeddings in a low-D space for illustration ────────────────
# In reality CLIP embeddings are 512-dim (ViT-B/32) or 768-dim (ViT-L/14)
CLIP_DIM = 512

def l2_normalize(v: np.ndarray) -> np.ndarray:
    """L2-normalize a vector (or batch of vectors)."""
    norm = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / (norm + 1e-8)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two normalized vectors."""
    a, b = l2_normalize(a), l2_normalize(b)
    return float(np.dot(a, b))

def clip_score(text_emb: np.ndarray, image_emb: np.ndarray, w: float = 2.5) -> float:
    """CLIPScore = w * max(0, cos(text_emb, image_emb))."""
    cos_sim = cosine_similarity(text_emb, image_emb)
    return w * max(0.0, cos_sim)

# ── Scenario: compare three generated images against a text prompt ────────────
# Text: "a golden retriever running on a beach"
prompt_emb = l2_normalize(np.random.randn(CLIP_DIM))   # "ground truth" text direction

# Simulate image embeddings with varying alignment
def make_image_emb(similarity_target: float, noise_scale: float = 0.3) -> np.ndarray:
    """Create an image embedding with target cosine similarity to the prompt."""
    aligned_component = similarity_target * prompt_emb
    noise = l2_normalize(np.random.randn(CLIP_DIM)) * np.sqrt(1 - similarity_target**2)
    emb = aligned_component + noise * noise_scale
    return l2_normalize(emb)

scenarios = [
    ("Perfect match\n(dog on beach)",    make_image_emb(0.92), '#2ecc71'),
    ("Partial match\n(dog, no beach)",   make_image_emb(0.65), '#f39c12'),
    ("Wrong subject\n(cat indoors)",     make_image_emb(0.15), '#e74c3c'),
    ("Negative space\n(blank image)",    make_image_emb(-0.05),  '#95a5a6'),
]

print("📊 CLIPScore Evaluation")
print(f"   {'Scenario':35s} | {'cos_sim':>8} | {'CLIPScore':>10}")
print("   " + "-" * 60)

scores = []
for name, img_emb, color in scenarios:
    cos = cosine_similarity(prompt_emb, img_emb)
    cs  = clip_score(prompt_emb, img_emb)
    scores.append((name, cos, cs, color))
    print(f"   {name.replace(chr(10), ' '):35s} | {cos:>8.4f} | {cs:>10.4f}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d1117')

names_short = [s[0].replace('\n', '\n') for s in scores]
cos_vals    = [s[1] for s in scores]
cs_vals     = [s[2] for s in scores]
colors_bar  = [s[3] for s in scores]

bars1 = ax1.barh(range(len(scores)), cos_vals, color=colors_bar, alpha=0.85, height=0.6)
ax1.set_yticks(range(len(scores)))
ax1.set_yticklabels(names_short, color='white', fontsize=9)
ax1.set_xlabel('Cosine Similarity', color='white')
ax1.set_title('Cosine Similarity (prompt ↔ image)', color='white', fontsize=10)
ax1.axvline(0, color='white', linewidth=0.5)
ax1.tick_params(colors='white')

bars2 = ax2.barh(range(len(scores)), cs_vals, color=colors_bar, alpha=0.85, height=0.6)
ax2.set_yticks(range(len(scores)))
ax2.set_yticklabels(names_short, color='white', fontsize=9)
ax2.set_xlabel('CLIPScore (× 2.5, clipped at 0)', color='white')
ax2.set_title('CLIPScore', color='white', fontsize=10)
ax2.axvline(0, color='white', linewidth=0.5)
ax2.tick_params(colors='white')

# Value labels
for bar, val in zip(bars2, cs_vals):
    ax2.text(max(0.05, val + 0.02), bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', color='white', fontsize=9)

fig.suptitle('🎯 CLIPScore: How Well Does the Image Match the Prompt?',
             color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/clipscore.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print()
print("✅ Negative cos similarity → CLIPScore = 0 (clamped, not negative)")
print("   A CLIPScore > 0.25 (w=2.5 * 0.10) is typically considered decent alignment.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Empirical quality vs. inference steps curve ───────────────────────────────
# Based on published results from Ho et al. (DDPM), Song et al. (DDIM),
# and community benchmarks on standard prompts.

# DDPM: quality keeps improving but needs many steps
ddpm_steps = np.array([10, 25, 50, 100, 250, 500, 1000])
ddpm_fid   = np.array([120, 55, 20, 8.5, 4.2, 3.5, 3.17])   # FID (lower = better)
ddpm_clip  = np.array([0.20, 0.24, 0.27, 0.30, 0.315, 0.320, 0.325])  # CLIPScore

# DDIM (deterministic, η=0): similar quality at far fewer steps
ddim_steps = np.array([10, 20, 30, 50, 100, 200])
ddim_fid   = np.array([14.0, 6.8, 5.1, 4.2, 3.8, 3.6])      # FID
ddim_clip  = np.array([0.28, 0.305, 0.315, 0.320, 0.322, 0.323])  # CLIPScore

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='white')

# ── FID vs Steps ──────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(ddpm_steps, ddpm_fid, 'o-', color='#e74c3c', lw=2.5, ms=7, label='DDPM')
ax.plot(ddim_steps, ddim_fid, 's-', color='#3498db', lw=2.5, ms=7, label='DDIM')
ax.set_xscale('log')
ax.set_xlabel('Number of inference steps (log scale)', color='white')
ax.set_ylabel('FID ↓ (lower = better)', color='white')
ax.set_title('FID vs. Inference Steps', color='white', fontsize=11)
ax.legend(facecolor='#1a1a2e', labelcolor='white')

# Annotate the sweet spots
ax.annotate('DDIM sweet spot\n(20-50 steps)', xy=(20, 6.8),
            xytext=(80, 30), color='#3498db', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#3498db'))
ax.annotate('DDPM needs 250+\nsteps for good FID', xy=(250, 4.2),
            xytext=(30, 15), color='#e74c3c', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))

# ── CLIPScore vs Steps ────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(ddpm_steps, ddpm_clip, 'o-', color='#e74c3c', lw=2.5, ms=7, label='DDPM')
ax.plot(ddim_steps, ddim_clip, 's-', color='#3498db', lw=2.5, ms=7, label='DDIM')
ax.set_xscale('log')
ax.set_xlabel('Number of inference steps (log scale)', color='white')
ax.set_ylabel('CLIPScore ↑ (higher = better)', color='white')
ax.set_title('CLIPScore vs. Inference Steps', color='white', fontsize=11)
ax.legend(facecolor='#1a1a2e', labelcolor='white')
ax.set_ylim(0.18, 0.35)

# Highlight the diminishing returns zone
axes[1].axhspan(0.315, 0.35, alpha=0.08, color='#2ecc71')
axes[1].text(200, 0.318, 'Diminishing returns\n(>50 steps)', color='#2ecc71', fontsize=8)

fig.suptitle('📈 Quality-Speed Tradeoff: DDPM vs DDIM (illustrative)',
             color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/quality_vs_steps.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print("📌 Key takeaways:")
print("   1. DDIM reaches DDPM-level FID with ~20× fewer steps")
print("   2. CLIPScore plateaus quickly — text alignment saturates early")
print("   3. FID keeps improving up to ~500 steps for DDPM (perceptual quality)")
print("   4. In production, DDIM @ 20-50 steps is the standard choice")

## 🏭 Data Pipeline — What Goes into Training a Text-to-Image Model?

The quality of a text-to-image model is 50% architecture, 50% data. Here's what the pipeline looks like at scale (e.g., LAION-5B → Stable Diffusion):

```
Raw internet crawl (billions of image-text pairs)
          │
    ① Deduplication
          │  • Perceptual hashing (pHash) to remove near-duplicate images
          │  • URL dedup + semantic dedup via embedding similarity
          │  • Why: duplicates cause memorization and inflate perceived dataset size
          │
    ② NSFW Detection
          │  • Classifier trained on known NSFW content
          │  • Conservative threshold → remove borderline content
          │  • Why: legal, ethical, prevents model from generating harmful content
          │
    ③ Aesthetic Scoring
          │  • LAION-Aesthetics: linear probe on CLIP embeddings trained
          │    on human preference data (AVA dataset)
          │  • Filter: keep top ~5-20% by aesthetic score
          │  • Why: training on low-quality images degrades output quality
          │
    ④ CLIP-Based Caption Quality Filtering
          │  • Compute CLIP similarity between image and alt-text
          │  • Keep pairs with similarity > threshold (e.g., 0.28)
          │  • Why: many web captions are SEO spam, not actual descriptions
          │
    ⑤ Caption Enhancement (optional)
          │  • Run a captioning model (e.g., CogVLM, BLIP-2) to generate
          │    richer captions than the original alt-text
          │  • Increases text-image alignment and instruction following
          │
    ⑥ Watermark Detection
          │  • Binary classifier for stock-photo watermarks
          │  • Remove or down-weight to avoid model generating fake watermarks
          │
    Final curated dataset (e.g., 600M → 5M high-quality pairs)
```

### Why Aggressive Filtering?

| Filter | % typically removed | Impact if skipped |
|--------|--------------------|-----------------|
| Dedup | 20-40% | Memorization, FID inflation |
| NSFW | 5-15% | Generates harmful content |
| Aesthetic | 80-95% | Blurry, low-quality outputs |
| CLIP similarity | 20-50% | Poor text-image alignment |

> 💡 **Staff-level insight**: "More data is better" is only true if quality is maintained. A 5M high-quality dataset often beats a 500M noisy dataset for text-image alignment.

## 🏗️ Inference System Design — From Prompt to Image

Here's the complete production pipeline for a text-to-image service:

```
User Input
   │
   ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  ① PROMPT SAFETY FILTER                                                  │
│     • Classifier for prohibited content (violence, CSAM, etc.)           │
│     • Block or reject immediately                                         │
│     • Latency budget: < 5ms (fast binary classifier)                     │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  ② PROMPT ENHANCEMENT (optional)                                         │
│     • LLM rewrites the prompt for better image quality                   │
│     • "a dog" → "a golden retriever, professional photography,           │
│        sharp focus, natural lighting, 4K"                                │
│     • Negative prompt injection: "blurry, low quality, watermark"        │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  ③ TEXT ENCODER                                                           │
│     • CLIP ViT-L/14 or T5-XXL                                            │
│     • Produces: text_emb (B, 77, 768)                                    │
│     • Cached per unique prompt (saves 10% compute)                       │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  ④ DIFFUSION MODEL (in latent space)                                     │
│     • VAE Encoder: image space (512×512×3) → latent space (64×64×4)     │
│     • U-Net or DiT: 20-50 DDIM steps with CFG (w=7.5)                   │
│     • VAE Decoder: latent (64×64×4) → image (512×512×3)                 │
│     • Running on A100/H100 GPU, quantized to FP16 or BF16               │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  ⑤ OUTPUT HARM DETECTION                                                 │
│     • Second safety pass on the generated image                          │
│     • NSFW classifier + watermark detector                               │
│     • Block if threshold exceeded                                        │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  ⑥ SUPER-RESOLUTION CASCADE (optional)                                   │
│     • 64×64 base model → 256×256 SR model → 1024×1024 SR model          │
│     • Each stage is a separate (smaller) diffusion model                 │
│     • DALL-E 2 uses this approach; SD uses latent upscaling              │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
                                 ▼
                        Final image → User 🖼️
```

### Latency Budget (realistic A100 numbers)

| Stage | Latency | Notes |
|-------|---------|-------|
| Safety filter | ~5ms | Fast MLP classifier |
| Text encoder | ~20ms | Cached = ~0ms |
| DDIM 20 steps | ~1.5s | Main bottleneck |
| SR cascade (2×) | ~1.0s | Optional |
| VAE decode | ~100ms | Fast |
| **Total** | **~2-3s** | **P90 production target** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import zoom

np.random.seed(5)

# ── Simulate the super-resolution cascade ────────────────────────────────────
# Generate a low-res 8×8 latent to represent the 64×64 latent image
# (we simulate small for display clarity)

# Step 1: 64×64 base image (noise → first coherent image)
base_64 = np.random.randn(64, 64) * 0.3
# Add a fake "subject" — a bright blob in the center
y, x = np.ogrid[-32:32, -32:32]
mask = x**2 + y**2 < 18**2
base_64[mask] = 0.8 + 0.2 * np.random.randn(mask.sum())
# Inner brighter region (face/detail)
inner = x**2 + y**2 < 8**2
base_64[inner] = 1.2 + 0.1 * np.random.randn(inner.sum())
base_64 = np.clip(base_64, 0, 1)

def sr_step(
    img: np.ndarray,
    scale: int = 4,
    noise_level: float = 0.05,
    detail_boost: float = 0.08,
) -> np.ndarray:
    """
    Simulated super-resolution step:
    1. Bicubic upscaling (what any SR model starts from)
    2. Add high-frequency details (what SR diffusion adds)
    3. Sharpen edges slightly

    Real SR models: run a separate diffusion model conditioned on the low-res image.
    """
    # Bicubic upsample
    upscaled = zoom(img, scale, order=3)
    H, W = upscaled.shape

    # Add fine-grained noise (high-frequency details that diffusion SR adds)
    detail_noise = np.random.randn(H, W) * detail_boost
    # Edge sharpening via unsharp mask (simplified)
    from scipy.ndimage import uniform_filter
    blurred = uniform_filter(upscaled, size=3)
    sharpened = upscaled + 0.5 * (upscaled - blurred)

    result = sharpened + detail_noise
    return np.clip(result, 0, 1)

# Run the cascade: 64 → 256 → 1024
# (display at smaller sizes to fit: 64, 256, 1024 → downsample for display)
img_64   = base_64                                          # 64×64
img_256  = sr_step(img_64, scale=4, detail_boost=0.06)     # 256×256
img_1024 = sr_step(img_256, scale=4, detail_boost=0.04)    # 1024×1024

print(f"Image resolutions in cascade:")
print(f"  Stage 1 (base):       {img_64.shape}")
print(f"  Stage 2 (first SR):  {img_256.shape}")
print(f"  Stage 3 (second SR): {img_1024.shape}")

# ── Visualize ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#1a1a2e')

# Display at native resolution (but limit 1024 to 512 for memory)
display_imgs = [
    (img_64,              f"64 × 64\n(Base diffusion model)",   '#e74c3c'),
    (img_256,             f"256 × 256\n(SR Model × 4)",          '#f39c12'),
    (img_1024[::2, ::2],  f"1024 × 1024\n(SR Model × 4 again)", '#2ecc71'),
]

for ax, (img, title, border_color) in zip(axes, display_imgs):
    ax.imshow(img, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(title, color='white', fontsize=11, pad=6)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)

# Add arrows between stages
for x_pos in [0.355, 0.665]:
    fig.text(x_pos, 0.5, '→\nSR\nModel', ha='center', va='center',
             color='white', fontsize=14, fontweight='bold')

fig.suptitle(
    '📐 Super-Resolution Cascade: 64×64 → 256×256 → 1024×1024\n'
    'Each stage runs a separate (smaller, faster) diffusion model',
    color='white', fontsize=12, fontweight='bold'
)

# Add pixel count annotation
fig.text(0.5, -0.04,
         f'Pixel counts: 4,096  →  65,536 (16×)  →  1,048,576 (256×)',
         ha='center', color='#aaa', fontsize=10)

plt.tight_layout()
plt.savefig('/tmp/sr_cascade.png', dpi=100, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print()
print("🔬 Why a cascade instead of directly generating 1024×1024?")
print("   • Memory: U-Net attention cost scales O(N²) with N = H×W")
print("   • At 1024×1024: N = 1,048,576 → attention matrix = 1T elements")
print("   • 64×64 base: N = 4,096 → 65,536 elements — 16,000× smaller")
print("   • SR models are conditioned on low-res → much easier task")
print()
print("   DALL-E 2 cascade: 64 → 256 → 1024")
print("   Imagen (Google): 64 → 256 → 1024")
print("   Stable Diffusion: works at 512 natively via latent compression")

## 📋 Interview Cheat Sheet

Everything you need to answer staff-level text-to-image questions in an interview. 🎯

---

### Classifier-Free Guidance (CFG)

| Question | Answer |
|----------|--------|
| What is CFG? | Extrapolates noise prediction away from unconditional and toward conditional at inference |
| Formula | ε̂ = ε_uncond + w·(ε_cond - ε_uncond) |
| Typical w | 7.5 for Stable Diffusion; 3-10 range depending on application |
| Training trick | Drop text conditioning 10% of the time during training |
| w=0 | Pure unconditional; w=1 = pure conditional; w>1 = amplified |
| Trade-off | High w → better prompt adherence but reduced diversity and artifacts |

---

### DDIM vs DDPM

| | DDPM | DDIM |
|-|------|------|
| **Paper** | Ho et al. 2020 | Song et al. 2021 |
| **Steps** | 1000 | 20-50 |
| **Stochastic?** | Yes (adds noise each step) | No (η=0 = deterministic) |
| **Reproducible?** | No (random noise) | Yes (same seed = same image) |
| **Quality** | Excellent | Very good (slight tradeoff) |
| **Key insight** | DDIM is a non-Markovian process that generalizes DDPM |

---

### U-Net vs DiT

| | U-Net | DiT |
|-|-------|-----|
| **Architecture** | CNN encoder-decoder + skip connections | Patchify → Transformer → Unpatchify |
| **Used in** | SD1/2, DALL-E 2 | SD3, FLUX, Sora |
| **Scales like** | No known scaling laws | LLMs (power law) |
| **Inductive bias** | Locality (good for textures) | Global (better composition) |
| **Speed** | Faster (same params) | Slower but catching up |

---

### CLIPScore

| Question | Answer |
|----------|--------|
| Formula | w · max(0, cos(e_text, e_image)), w=2.5 |
| What it measures | Text-image semantic alignment |
| Limitation | Can't count objects, spatial relations, temporal order |
| Used alongside | FID (quality vs. reference), human eval |
| Good score | > 0.25 generally indicates decent alignment |

---

### System Design Quick Reference

```
Prompt → [Safety] → [Enhance] → [Text Encoder] → [Latent Diffusion]
       → [VAE Decode] → [Output Safety] → [Super-Resolution] → Image
```

- **Latency**: ~2-3s on A100 for 1024×1024 with cascade
- **Scaling**: Batch requests, quantize to FP16/BF16, compile with torch.compile
- **Caching**: Cache text embeddings per unique prompt
- **Quality knobs**: CFG scale, num_inference_steps, negative prompt, scheduler choice

## 📝 Quiz — Chapter 09, Part 2

Test yourself! Try to answer before revealing. 💪

---

**Q1: What does guidance scale w=0 produce, and what does w=15 produce?**

<details><summary>💡 Reveal Answer</summary>

**w=0**: The formula reduces to ε_uncond, so you get purely unconditional generation — the model ignores the text prompt entirely and generates "average" images from the data distribution.

**w=15**: The model strongly over-amplifies the conditional direction. Images are hyper-saturated, highly specific to the prompt, but often look artificial, over-contrasted, or bizarre ("deep frying" the image).

</details>

---

**Q2: Why does DDIM only need 20 steps while DDPM needs 1000?**

<details><summary>💡 Reveal Answer</summary>

DDIM reframes the reverse process as a non-Markovian process where each step can make a larger "jump" by directly predicting x₀ and then re-noising to the previous timestep. Because the step is deterministic (η=0), it doesn't accumulate error from stochastic noise. DDIM can skip many intermediate timesteps and still converge to a high-quality image.

</details>

---

**Q3: What are two limitations of CLIPScore as an evaluation metric?**

<details><summary>💡 Reveal Answer</summary>

1. **Counting failure**: CLIP can't verify quantity. "Three cats" and "two cats" may score similarly because CLIP embeds the bag-of-words meaning, not the exact count.
2. **Spatial/relational failure**: "A dog to the LEFT of a cat" is hard to distinguish from "A cat to the LEFT of a dog" for CLIP, since it encodes global semantics rather than fine spatial structure.

</details>

---

**Q4: Why does Stable Diffusion generate images in "latent space" rather than pixel space?**

<details><summary>💡 Reveal Answer</summary>

A VAE compresses 512×512×3 images (~786K dims) into 64×64×4 latents (~16K dims) — a ~50× compression. The diffusion process runs in this lower-dimensional space, so the U-Net processes much smaller feature maps. This reduces memory and compute dramatically while preserving perceptual quality (the VAE handles the fine pixel details separately).

</details>

---

**Q5: In an interview, you're asked to design a text-to-image service that must serve 10,000 requests/second. What are the top 3 optimizations you'd implement?**

<details><summary>💡 Reveal Answer</summary>

1. **Step reduction**: Use DDIM with 20 steps instead of DDPM 1000 steps. This gives ~50× speedup in model forward passes (the dominant cost).

2. **Batching + quantization**: Batch multiple requests per GPU call (increase GPU utilization). Quantize the model to FP16/BF16 — 2× memory reduction means fitting larger batches, with minimal quality loss.

3. **Text embedding caching**: CLIP text embeddings are deterministic per prompt. Cache them in a key-value store (Redis). Popular prompts like "a cute puppy" are requested constantly — serve cached embeddings instantly, skip the encoder entirely.

Bonus: `torch.compile()` the model for kernel fusion; use specialized attention kernels (Flash Attention) to speed up cross-attention by ~2×.

</details>